In [ ]:
!pip install -U langchain
!pip install -U langchain-community
!pip install -U langchain-classic
!pip install -U langchain-text-splitters
!pip install -U langchain-huggingface
!pip install -U langchain-chroma
!pip install -U chromadb
!pip install -U sentence-transformers

In [ ]:
!pip install -U langchain-nvidia-ai-endpoints

In [52]:
from langchain_classic.storage import LocalFileStore, create_kv_docstore
import os, re, base64
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever, BM25Retriever
from langchain_classic.storage import InMemoryStore
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage
from langchain_classic.schema import Document
from sentence_transformers import CrossEncoder

import os
import getpass
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser




In [27]:
local_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cpu'}, # Use 'cuda' if you have an NVIDIA GPU
    encode_kwargs={'normalize_embeddings': True}
)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2221.72it/s, Materializing param=pooler.dense.weight]                               


In [28]:
persist_dir = "./chroma_db_local/chroma_db_local"
parent_store_dir = "./parent_store_local/parent_store_local"

# 2. Create the Parent Storage (This makes the text blocks persistent!)
fs = LocalFileStore(parent_store_dir)
store = create_kv_docstore(fs)

# 3. Setup Chroma as before
vectorstore = Chroma(
    collection_name="med_rag_local",
    embedding_function=local_embeddings,
    persist_directory=persist_dir
)


child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
# 4. Initialize Retriever with the PERSISTENT store
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter
)

In [29]:
def query_textbook(question):
    print(f"🔍 Searching: {question}\n" + "="*50)
    docs = retriever.invoke(question)

    for i, doc in enumerate(docs[:5]):
        print(f"RESULT {i+1} | {doc.metadata['chapter']} | Section {doc.metadata['section']}")
        print(f"🖼️ Base64 Image Metadata Found: {'image_base64' in doc.metadata}")
        print(f"📝 TEXT:\n{doc.page_content[:400]}...\n")

query_textbook("What is Humanism?")

🔍 Searching: What is Humanism?
RESULT 1 | Personality | Section 11.5
🖼️ Base64 Image Metadata Found: False
📝 TEXT:
# <span id="page-38-0"></span>**11.5 Humanistic Approaches**  
#### **LEARNING OBJECTIVES**  
By the end of this section, you will be able to:  
• Discuss the contributions of Abraham Maslow and Carl Rogers to personality development  
As the "third force" in psychology, humanism is touted as a reaction both to the pessimistic determinism of psychoanalysis, with its emphasis on psychological distu...

RESULT 2 | Introduction to Psychology | Section 1.2
🖼️ Base64 Image Metadata Found: False
📝 TEXT:
### **Maslow, Rogers, and Humanism**  
During the early 20th century, American psychology was dominated by behaviorism and psychoanalysis. However, some psychologists were uncomfortable with what they viewed as limited perspectives being so influential to the field. They objected to the pessimism and determinism (all actions driven by the unconscious) of Freud. They also disliked

In [ ]:

# 1. Secure NVIDIA API Key
if "NVIDIA_API_KEY" not in os.environ:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA NIM API Key: ")

# 2. Initialize the Best Free LLM (Llama 3.3 70B Instruct)
print("🚀 Initializing Llama-3.3-70B via NVIDIA API...")
llm = ChatNVIDIA(
    model="meta/llama-3.3-70b-instruct",
    temperature=0.2, # Keep it low for factual textbook answers
    max_tokens=1024
)

# 3. Create the System Prompt
# This tells the LLM exactly how to behave and use your metadata
prompt_template = """
You are an expert medical and psychology tutor. Answer the user's question based ONLY on the provided textbook context.
If the answer is not in the context, say "I cannot find the answer in the textbook."

Here is the retrieved context from the textbook:
{context}

Question: {question}

Answer the question clearly and concisely. If the context includes a chapter or section title, feel free to cite it!

write the citiation in this format: [Source: Chapter X, Section Y] at the end of the answer.
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

# 4. Helper Function to Format Retrieved Documents
def format_docs(docs):
    formatted_text = []
    for doc in docs:
        # We inject your hard-earned metadata directly into the LLM's context!
        header = f"[Source: Chapter {doc.metadata.get('chapter', 'Unknown')}, Section {doc.metadata.get('section', 'Unknown')}]"
        formatted_text.append(f"{header}\n{doc.page_content}")
    return "\n\n".join(formatted_text)

# 5. Build the Final RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Generative RAG Chain Ready!")

🚀 Initializing Llama-3.3-70B via NVIDIA API...
✅ Generative RAG Chain Ready!


/tmp/ipykernel_29051/3604520867.py:14: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(


In [31]:
question = "Who is Binary Search?"
print(f"👤 You: {question}")
print("🤖 Llama 3.3: Thinking...\n")

# Run the chain!
answer = rag_chain.invoke(question)

print(answer)

👤 You: Who is Binary Search?
🤖 Llama 3.3: Thinking...

I cannot find the answer in the textbook. 

[Source: Chapter Sensation and Perception, Section 5.6], [Source: Chapter Industrial-Organizational Psychology, Section 13.1], [Source: Chapter Therapy and Treatment, Section 16.5]


## Implementing Hyde (Hypothethical Document Embedding)

In [ ]:
    def generate_hyde_vector(query: str, llm_client, embedding_model) -> list[float]:
        """
        Generates a hypothetical document and returns its vector embedding.
        
        Args:
            query: The user's search query.
            llm_client: Your initialized LLM (e.g., your Llama generator function).
            embedding_model: Your initialized BGE-M3 embedder.
        """
        
        # 1. The Prompt: Instruct the LLM to write a textbook-style answer
        prompt = f"""Given the following question, write a hypothetical, detailed textbook passage that directly answers it. 
        Write in an academic tone, use standard formatting, and include relevant domain keywords that might appear in a textbook.
        
        Question: {query}
        
        Hypothetical Textbook Passage:"""
        
        # 2. Generate the hypothetical document
        # (Adjust the calling method based on your specific LLM wrapper)

        response = llm_client.invoke([
            HumanMessage(content=prompt)
        ])
        
        hypothetical_doc = response.content
        print(f"--- Hypothetical Document Generated ---\n{hypothetical_doc}\n---------------------------------------")
        
        # 3. Embed the generated document instead of the raw query
        # (Adjust .encode() based on whether you are using HuggingFace, SentenceTransformers, etc.)
        hyde_vector = embedding_model.embed_query(hypothetical_doc)
        
        return hyde_vector

    # --- Usage Example ---
    # user_query = "What are the prigenerate_hyde_vectormary functions of the sympathetic nervous system?"
    # query_vector = (user_query, my_llama_model, my_bge_m3_model)
    # 
    # # Now you pass `query_vector` into your Vector DB for the similarity search!

In [33]:
query = "What are the basic parts of a neuron?"
hyde_vector = generate_hyde_vector(query, llm, local_embeddings);


--- Hypothetical Document Generated ---
**Neuroanatomy: The Basic Structure of Neurons**

The neuron, also known as the nerve cell, is the fundamental unit of the nervous system. It is a specialized cell designed to transmit and process information through electrical and chemical signals. The basic structure of a neuron consists of several distinct parts, each with a unique function.

### **Dendrites**

The dendrites are the branching extensions of the neuron that receive signals from other neurons. They are responsible for collecting and integrating synaptic inputs from multiple sources, allowing the neuron to process complex information. Dendrites are covered with small protrusions called dendritic spines, which increase the surface area available for synaptic transmission. The dendrites are the primary site of excitatory and inhibitory synaptic inputs, which are mediated by neurotransmitters such as glutamate and GABA (gamma-aminobutyric acid).

### **Cell Body (Soma)**

The cell bo

## Define and Implementing the Hybrid Search and Vector Search
    

In [35]:
def retrieve_and_fuse(query: str, hyde_vector: list[float], bm25_retriever, vector_store, top_k: int = 5) -> list:
    """
    Executes a hybrid search and fuses the results using Reciprocal Rank Fusion (RRF).
    
    Args:
        query: The raw user string (for exact keyword matching).
        hyde_vector: The embedded hypothetical document (for semantic matching).
        bm25_retriever: Your initialized LangChain BM25Retriever.
        vector_store: Your initialized LangChain vector database.
        top_k: Number of initial documents to retrieve from each source.
    """
    
    # ---------------------------------------------------------
    # STEP 2: DUAL-STREAM RETRIEVAL
    # ---------------------------------------------------------
    
    # A. Keyword Search (BM25) - Uses the exact user query
    # We override the default k to ensure we get exactly top_k results
    bm25_retriever.k = top_k 
    bm25_docs = bm25_retriever.invoke(query)
    
    # B. Vector Search - Uses the dense HyDE vector
    # similarity_search_by_vector allows us to bypass the text embedding step
    vector_docs = vector_store.similarity_search_by_vector(hyde_vector, k=top_k)
    
    print(f"Retrieved {len(bm25_docs)} BM25 docs and {len(vector_docs)} Vector docs.")

    # ---------------------------------------------------------
    # STEP 3: RECIPROCAL RANK FUSION (RRF)
    # ---------------------------------------------------------
    
    fused_scores = {}
    rrf_k = 60 # Standard constant used in the RRF formula
    
    def score_docs(docs):
        for rank, doc in enumerate(docs):
            # We use the chunk's text as the unique identifier.
            # (If you added a 'chunk_id' to your metadata during chunking, use doc.metadata['chunk_id'] instead!)
            doc_id = doc.page_content 
            
            if doc_id not in fused_scores:
                fused_scores[doc_id] = {"doc": doc, "score": 0.0}
            
            # The RRF math: score = 1 / (rank + k)
            fused_scores[doc_id]["score"] += 1.0 / (rank + rrf_k)

    # Score both lists
    score_docs(bm25_docs)
    score_docs(vector_docs)
    
    # Sort the fused dictionary by the calculated RRF score in descending order
    reranked_results = sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)
    
    # Extract just the Document objects from the sorted list
    final_fused_docs = [item["doc"] for item in reranked_results]
    
    print(f"Successfully fused into {len(final_fused_docs)} unique child chunks.")
    
    return final_fused_docs

# --- Usage Example ---
# fused_child_chunks = retrieve_and_fuse(user_query, query_vector, my_bm25, my_vector_db)

In [47]:


# Extract all child chunks directly from Chroma
# Calling .get() provides a dictionary output containing the 'documents' and 'metadatas' arrays.
chroma_data = vectorstore.get()

# Reconstruct the LangChain Document objects so BM25 can read them
child_chunks = []
for i in range(len(chroma_data['documents'])):
    doc = Document(
        page_content=chroma_data['documents'][i],
        metadata=chroma_data['metadatas'][i] if chroma_data['metadatas'] else {}
    )
    child_chunks.append(doc)

print(f"Loaded {len(child_chunks)} child chunks directly from ChromaDB.")

# 3. Instantiate the BM25Retriever using the extracted chunks
my_bm25_retriever = BM25Retriever.from_documents(child_chunks)
fused_child_chunk =  retrieve_and_fuse(query,hyde_vector,my_bm25_retriever,vectorstore, top_k=5)
print("\n--- Top Fused Chunk ---")
print(fused_child_chunk[0].page_content)

Loaded 7633 child chunks directly from ChromaDB.
Retrieved 5 BM25 docs and 5 Vector docs.
Successfully fused into 9 unique child chunks.

--- Top Fused Chunk ---
# <span id="page-39-0"></span>**3.2 Cells of the Nervous System**  
## **LEARNING OBJECTIVES**  
By the end of this section, you will be able to:  
- Identify the basic parts of a neuron
- Describe how neurons communicate with each other
- Explain how drugs act as agonists or antagonists for a given neurotransmitter system


In [48]:
for i, doc in enumerate(fused_child_chunk):
    print(f"RESULT {i+1} | {doc.metadata['chapter']} | Section {doc.metadata['section']}")
    print(f"🖼️ Base64 Image Metadata Found: {'image_base64' in doc.metadata}")
    print(f"📝 TEXT:\n{doc.page_content}...\n")

RESULT 1 | Biopsychology | Section 3.2
🖼️ Base64 Image Metadata Found: False
📝 TEXT:
# <span id="page-39-0"></span>**3.2 Cells of the Nervous System**  
## **LEARNING OBJECTIVES**  
By the end of this section, you will be able to:  
- Identify the basic parts of a neuron
- Describe how neurons communicate with each other
- Explain how drugs act as agonists or antagonists for a given neurotransmitter system...

RESULT 2 | Biopsychology | Section 3.2
🖼️ Base64 Image Metadata Found: False
📝 TEXT:
The nucleus of the neuron is located in the **soma**, or cell body. The soma has branching extensions known as **dendrites**. The neuron is a small information processor, and dendrites serve as input sites where signals are received from other neurons. These signals are transmitted electrically across the soma and down a major extension from the soma known as the **axon**, which ends at multiple **terminal buttons**. The terminal buttons contain **synaptic vesicles** that house...

RESULT 3 | Sen

## Implementing the cross-encoder re-ranker from BGE Ecosystem bge-reranker-large


In [ ]:

# 1. Initialize the Cross-Encoder model
# (Note: This will download the model the first time you run it. Switch to 'BAAI/bge-reranker-base' if 'large' is too heavy for your RAM limits)
bge_reranker = CrossEncoder("BAAI/bge-reranker-large", device="cpu")

def rerank_chunks(query: str, fused_docs: list, reranker_model, top_n: int = 3) -> list:
    """Passes the RRF fused docs through BGE-Reranker for precise scoring."""
    
    # The CrossEncoder expects a list of pairs: [[query, doc1], [query, doc2], ...]
    pairs = [[query, doc.page_content] for doc in fused_docs]
    
    # Predict similarity scores for each pair
    scores = reranker_model.predict(pairs)
    
    # Zip the scores with the documents and sort them highest to lowest
    scored_docs = list(zip(scores, fused_docs))
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    
    # Return strictly the top N child chunks
    return [doc for score, doc in scored_docs[:top_n]]

# 2. Execute the re-ranking on your fused chunks!
# We pass in your 'query' variable and the 'fused_child_chunk' list you just generated
top_child_chunks = rerank_chunks(query, fused_child_chunk, bge_reranker, top_n=3)

print("\n--- Absolute Top Re-ranked Chunk ---")
print(top_child_chunks[0].page_content)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1614.66it/s, Materializing param=roberta.encoder.layer.23.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Absolute Top Re-ranked Chunk ---
The nucleus of the neuron is located in the **soma**, or cell body. The soma has branching extensions known as **dendrites**. The neuron is a small information processor, and dendrites serve as input sites where signals are received from other neurons. These signals are transmitted electrically across the soma and down a major extension from the soma known as the **axon**, which ends at multiple **terminal buttons**. The terminal buttons contain **synaptic vesicles** that house


In [53]:
for i, doc in enumerate(top_child_chunks):
    print(f"TOP {i+1} | {doc.metadata['chapter']} | Section {doc.metadata['section']}")
    print(f"🖼️ Base64 Image Metadata Found: {'image_base64' in doc.metadata}")
    print(f"📝 TEXT:\n{doc.page_content}...\n")

TOP 1 | Biopsychology | Section 3.2
🖼️ Base64 Image Metadata Found: False
📝 TEXT:
The nucleus of the neuron is located in the **soma**, or cell body. The soma has branching extensions known as **dendrites**. The neuron is a small information processor, and dendrites serve as input sites where signals are received from other neurons. These signals are transmitted electrically across the soma and down a major extension from the soma known as the **axon**, which ends at multiple **terminal buttons**. The terminal buttons contain **synaptic vesicles** that house...

TOP 2 | Biopsychology | Section 3.2
🖼️ Base64 Image Metadata Found: False
📝 TEXT:
Glia and neurons are the two cell types that make up the nervous system. While glia generally play supporting roles, the communication between neurons is fundamental to all of the functions associated with the nervous system. Neuronal communication is made possible by the neuron's specialized structures. The soma contains the cell nucleus, and the